# SLM — Google Colab GPU Training

**Before running:** Go to `Runtime → Change runtime type → GPU` (T4 is free; A100 with Colab Pro).

This notebook:
1. Mounts your Google Drive (checkpoints persist across sessions)
2. Uploads your local project **or** clones from a GitHub repo
3. Installs dependencies
4. Runs training with `device=cuda` and `bfloat16` automatically
5. Syncs checkpoints back to Drive every `save_every` steps


In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('⚠  No GPU detected! Go to Runtime → Change runtime type → GPU')
    sys.exit(1)

import torch
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  bfloat16 support: {torch.cuda.is_bf16_supported()}')

In [ ]:
# ── 2. Mount Google Drive ─────────────────────────────────────────────────────
# Checkpoints and logs will be saved here so they survive runtime resets.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT = '/content/drive/MyDrive/slm_training'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/logs', exist_ok=True)
print(f'Drive project dir: {DRIVE_PROJECT}')

In [ ]:
# ── 3a. Option A: Clone from GitHub ──────────────────────────────────────────
# Fill in your repo URL. If the repo is private, use a personal access token:
#   GITHUB_REPO = 'https://<TOKEN>@github.com/<user>/slm.git'

USE_GITHUB = False          # ← set True to clone from GitHub instead of uploading
GITHUB_REPO = 'https://github.com/YOUR_USER/slm.git'   # ← edit this

PROJECT_DIR = '/content/slm'

if USE_GITHUB:
    if os.path.isdir(PROJECT_DIR):
        !git -C {PROJECT_DIR} pull
    else:
        !git clone {GITHUB_REPO} {PROJECT_DIR}
    print('Cloned/pulled repo.')
else:
    print('Skipped GitHub clone — upload your files in the next cell.')

In [ ]:
# ── 3b. Option B: Upload project files from your computer ────────────────────
# Run this cell ONLY if USE_GITHUB = False above.
# It opens a file picker — select all .py files + tokenizer/ folder.

if not USE_GITHUB:
    from google.colab import files
    os.makedirs(PROJECT_DIR, exist_ok=True)
    os.chdir(PROJECT_DIR)
    print('Upload your project files (config.py, model.py, train.py, etc.):')
    uploaded = files.upload()
    print(f'Uploaded: {list(uploaded.keys())}')
else:
    os.chdir(PROJECT_DIR)

In [ ]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
# Colab already has torch; we just need the HuggingFace stack.
!pip install -q tokenizers>=0.19.0 datasets>=2.18.0 huggingface-hub>=0.22.0 psutil
print('Dependencies installed.')

In [ ]:
# ── 5. Configure paths to use Drive ──────────────────────────────────────────
# We symlink the checkpoint and log dirs into Drive so data persists.
import os, subprocess

os.chdir(PROJECT_DIR)

for sub in ['checkpoints', 'logs']:
    local_path = os.path.join(PROJECT_DIR, sub)
    drive_path = os.path.join(DRIVE_PROJECT, sub)
    if os.path.islink(local_path):
        os.unlink(local_path)
    elif os.path.isdir(local_path):
        # Move existing local checkpoints to drive first
        subprocess.run(['cp', '-rn', local_path + '/', drive_path + '/'], check=False)
        subprocess.run(['rm', '-rf', local_path], check=True)
    os.symlink(drive_path, local_path)
    print(f'Linked {local_path} → {drive_path}')

# If you have a pre-trained tokenizer in Drive, symlink it too
DRIVE_TOKENIZER = os.path.join(DRIVE_PROJECT, 'tokenizer')
LOCAL_TOKENIZER = os.path.join(PROJECT_DIR, 'tokenizer')
if os.path.isdir(DRIVE_TOKENIZER) and not os.path.exists(LOCAL_TOKENIZER):
    os.symlink(DRIVE_TOKENIZER, LOCAL_TOKENIZER)
    print(f'Linked tokenizer from Drive')

In [ ]:
# ── 6. (Optional) Train tokenizer if not already done ────────────────────────
import os
tokenizer_json = os.path.join(PROJECT_DIR, 'tokenizer', 'tokenizer.json')
if not os.path.exists(tokenizer_json):
    print('Training tokenizer (runs once, ~10-20 min on Colab)...')
    !python {PROJECT_DIR}/tokenizer_train.py
    # Copy trained tokenizer to Drive for future sessions
    !cp -r {PROJECT_DIR}/tokenizer {DRIVE_PROJECT}/
    print('Tokenizer saved to Drive.')
else:
    print('Tokenizer found — skipping.')

In [ ]:
# ── 7. Set HuggingFace token (optional but recommended for faster downloads) ──
import os

# Either paste your token here, or store it as a Colab secret:
#   Left panel → 🔑 Secrets → Add secret  (name: HF_TOKEN)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    if HF_TOKEN:
        os.environ['HF_TOKEN'] = HF_TOKEN
        print('HF_TOKEN loaded from Colab secrets.')
except Exception:
    # Fallback: set manually
    # os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
    print('No HF_TOKEN secret found — continuing without it (slower downloads).')

In [ ]:
# ── 8. Override config for Colab GPU ─────────────────────────────────────────
# These are patched at runtime without editing config.py.
# On a T4 (16 GB VRAM) the batch below uses ~9 GB — safe.
# On an A100 (40 GB) you can double batch_size or grad_accum_steps.

import sys
sys.path.insert(0, PROJECT_DIR)

from config import train_cfg, model_cfg

# GPU-optimised overrides
train_cfg.device           = 'cuda'
train_cfg.dtype            = 'auto'      # → bfloat16 automatically
train_cfg.batch_size       = 16          # larger batch on GPU
train_cfg.grad_accum_steps = 8           # effective batch = 128 sequences
train_cfg.num_workers      = 2
train_cfg.resume           = True        # auto-resume from Drive checkpoints

print(f'Model:  d_model={model_cfg.d_model}  n_layers={model_cfg.n_layers}  n_heads={model_cfg.n_heads}')
print(f'Batch:  {train_cfg.batch_size} x {train_cfg.grad_accum_steps} = {train_cfg.batch_size * train_cfg.grad_accum_steps} seqs/step')
print(f'Device: {train_cfg.device}  dtype: {train_cfg.dtype}')

In [ ]:
# ── 9. Start training ─────────────────────────────────────────────────────────
# Runs in-process so you can see live progress in the cell output.
# If the Colab session disconnects, just re-run from cell 1 —
# the checkpoint in Drive will be picked up automatically.

import sys
sys.path.insert(0, PROJECT_DIR)

from train import train
train()

In [ ]:
# ── 10. (Optional) Copy final checkpoint back to Drive manually ───────────────
# Not needed if symlinks worked in step 5, but useful as a safety backup.
!cp -r {PROJECT_DIR}/checkpoints/* {DRIVE_PROJECT}/checkpoints/ 2>/dev/null && echo 'Synced.' || echo 'Nothing to sync.'
!ls -lh {DRIVE_PROJECT}/checkpoints/ | tail -5

## Tips

| Colab tier | GPU | Effective batch (recommended) | Expected tokens/s |
|---|---|---|---|
| Free | T4 16 GB | batch=16, accum=8 → 128 seqs/step | ~800 k tok/s |
| Colab Pro | A100 40 GB | batch=32, accum=8 → 256 seqs/step | ~3 M tok/s |

- **Reconnects**: Colab free sessions disconnect after ~1.5 h of inactivity. The Drive symlink means you never lose progress — just rerun all cells.
- **Speed vs local**: A T4 is ~15× faster than CPU training for this model size.
- **bfloat16**: Automatically enabled on GPU via `dtype='auto'` in config. This halves VRAM and speeds up matmuls ~2× with no meaningful accuracy loss.
- **Larger model**: On A100 you can safely increase `model_cfg.d_model=384, n_layers=12` for a ~15M param model.
